# Rally E2B Two-Stage SFT

Trains the Gemma 4 E2B Heretic Rally RP candidate on Kaggle using the Alkahest v8 two-stage RP mix. Outputs stay under `/kaggle/working/rally-e2b-two-stage-sft` for the export notebook to consume as a kernel source.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        print(f'gpu_{i}=', props.name, round(props.total_memory / 1024**3, 2), 'GiB', f'sm_{major}{minor}')
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        if major < 7:
            raise RuntimeError('Kaggle assigned a pre-Volta GPU. Push with --accelerator NvidiaTeslaT4.')
except Exception as exc:
    print('torch_probe_error=', repr(exc))
    if 'pre-Volta' in str(exc):
        raise


In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

packages = [
    'git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af',
    'accelerate>=1.13.0',
    'bitsandbytes>=0.49.0',
    'peft>=0.19.0',
    'datasets>=4.8.0',
    'trl==0.23.1',
    'unsloth[cu128-torch280] @ git+https://github.com/unslothai/unsloth.git',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
    'sentencepiece>=0.2.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])


In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path(os.environ.get('RALLY_TWO_STAGE_WORK_DIR', '/kaggle/working/rally-e2b-two-stage-sft'))
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_rally_e2b_two_stage_sft.py'),
    '--work-dir', str(WORK_DIR),
    '--model-name', os.environ.get('RALLY_MODEL_NAME', 'p-e-w/gemma-4-E2B-it-heretic-ara'),
    '--stage-a-max-steps', os.environ.get('RALLY_STAGE_A_MAX_STEPS', '300'),
    '--stage-b-max-steps', os.environ.get('RALLY_STAGE_B_MAX_STEPS', '450'),
    '--max-seq-length', os.environ.get('RALLY_MAX_SEQ_LENGTH', '2048'),
    '--gradient-accumulation-steps', os.environ.get('RALLY_GRAD_ACCUM', '4'),
    '--learning-rate', os.environ.get('RALLY_STAGE_A_LR', '2e-4'),
    '--stage-b-learning-rate', os.environ.get('RALLY_STAGE_B_LR', '2e-4'),
]
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
import json, os

WORK_DIR = Path(os.environ.get('RALLY_TWO_STAGE_WORK_DIR', '/kaggle/working/rally-e2b-two-stage-sft'))
for path in sorted(WORK_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(WORK_DIR), round(path.stat().st_size / 1024**2, 2), 'MB')
report_path = WORK_DIR / 'rally-e2b-two-stage-sft-report.json'
print('report_path=', report_path)
if report_path.exists():
    print(report_path.read_text()[:4000])
